Save as: module4-simulation-lab.ipynb

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/moncap/micro-course/blob/main/module-04/module4-simulation-lab.ipynb)

# Lab 4 — Silicon Workers Under Three Contracts
### Gift exchange vs. the textbook
*Microeconomic Theory for Experimentalists & Applied Microeconomists*

**Research question:** Does contract form (fixed wage vs. piece rate)
shift silicon effort the way it shifts human effort — and when the flat
wage is *generous*, do LLM workers reciprocate like Fehr, Kirchsteiger &
Riedl's (1993) humans, or shirk like the textbook's?

**The task:** the worker chooses effort 0–10. Effort costs money per the
posted table ($c(e) = e^2/4$). Three contracts:

| Contract | Pay | Selfish benchmark | Gift-exchange pattern |
|---|---|---|---|
| `flat_low` | \$10 regardless | effort ≈ 0 | low effort |
| `flat_generous` | \$25 regardless | effort ≈ 0 | **high effort** |
| `piece_rate` | \$2.50 per unit | effort = 5 | effort ≈ 5 |

**Benchmarks:** the selfish optimum (effort 0 / 0 / 5) AND the human
gift-exchange pattern (effort rising in flat-wage generosity — FKR 1993;
fading over a field workday — Gneezy & List 2006). The model has read
both literatures: **matching "both" is the puzzle, not the success**
(PS4 Part C).

**Hypotheses to state before running (see handout):** H1 piece-rate
effort near 5; H2 flat-generous effort exceeds flat-low; H3 the
gift-exchange gap varies by persona (commit to a direction first).

**How to run:** no prior Python experience needed. Run the cells top to
bottom. Your required modification (handout) goes **only** in the cells
marked **CHANGE HERE**.


## Setup and the DRY_RUN switch

The notebook starts in **DRY_RUN mode**: it executes end-to-end on synthetic
placeholder data, with no API key and no cost, so you can check your design
and see the analysis pipeline work.

> ⚠️ **DRY_RUN output is fake.** It is generated by `mock_response()`, not
> by a model. Never report it as a result.

When your design is ready, set `DRY_RUN = False` in the cell below and run
again in Colab — you will be asked for your Anthropic API key via a hidden
prompt (the key is never stored in the notebook file).


In [ ]:
%pip install anthropic pandas matplotlib --quiet

In [ ]:
import itertools
import json
import random
import re
import time

import pandas as pd

DRY_RUN = True   # <-- set to False in Colab to query the real model

MODEL = "claude-sonnet-5"   # swap models here to test robustness across models
TEMPERATURE = 1.0            # fixed and reported — part of the experimental design
N_PER_CELL = 30              # decisions per persona x treatment cell

if DRY_RUN:
    print("=" * 62)
    print("DRY RUN: synthetic placeholder data — NOT model behavior.")
    print("Set DRY_RUN = False to run the real experiment.")
    print("=" * 62)

    class _MockBlock:
        def __init__(self, text):
            self.text = text

    class _MockResponse:
        def __init__(self, text):
            self.content = [_MockBlock(text)]

    class _MockMessages:
        def create(self, **kwargs):
            # mock_response() is defined in the design cell below
            return _MockResponse(mock_response(kwargs["messages"][0]["content"]))

    class _MockClient:
        messages = _MockMessages()

    client = _MockClient()
else:
    from getpass import getpass

    import anthropic

    # Never hard-code API keys. getpass keeps the key out of the notebook file.
    API_KEY = getpass("Paste your Anthropic API key: ")
    client = anthropic.Anthropic(api_key=API_KEY)


## Experimental design — CHANGE HERE (all four blocks live in this cell)

3 personas × 3 contracts = 9 cells × 30 agents = 270 calls.

Personas are **deliberately neutral**: a persona told to be reciprocal
that then reciprocates proves nothing. We describe workers with mild,
realistic dispositions and let the contract frame do the work.

Your required modification will usually mean **(1)** adding a factor
(e.g. `"wage_setter": ["employer", "computer"]` for the intentions arm,
or an hour-six sentence), **(2)** weaving it into `build_prompt`
*without changing any other wording across conditions*, and
**(3)** touching `parse_response` / `mock_response` only if the answer
format changes.


In [ ]:
PERSONAS = {                                     # CHANGE HERE (1 of 4)
    "baseline": (
        "You are an adult who just started a one-day temp job."
    ),
    "money_focused": (
        "You are an adult who just started a one-day temp job. You took "
        "it purely for the money and have no connection to this employer."
    ),
    "community_minded": (
        "You are an adult who just started a one-day temp job. You "
        "generally believe in doing right by people you deal with."
    ),
}

CONTRACTS = {
    "flat_low": "Your employer pays you a fixed $10 for the day, regardless of output.",
    "flat_generous": "Your employer pays you a fixed $25 for the day, regardless of output.",
    "piece_rate": "Your employer pays you $2.50 for every unit you produce.",
}

PERSONA_GRID = {
    "persona": list(PERSONAS),
    "contract": list(CONTRACTS),
}

# Effort cost table: c(e) = e^2 / 4, shown to the agent in full.
COST_TABLE = {e: e * e / 4 for e in range(11)}


def build_prompt(persona: dict) -> str:          # CHANGE HERE (2 of 4)
    """One effort decision. Design rules: persona first, situation second,
    response format last; identical wording across conditions except the
    contract sentence; no words like 'fair', 'generous', or 'deserve' —
    the wage levels must do the talking."""
    cost_lines = "\n".join(
        f"  effort {e}: costs you ${c:.2f}" for e, c in COST_TABLE.items())

    return (
        f"{PERSONAS[persona['persona']]}\n\n"
        f"{CONTRACTS[persona['contract']]}\n\n"
        "You now privately choose your effort for the day: a whole number "
        "from 0 to 10. Higher effort produces more units for the employer "
        "(one unit per effort level) but costs you personally, per this "
        "table:\n\n"
        f"{cost_lines}\n\n"
        "Your employer cannot observe your effort choice, only total "
        "output at day's end. There are no future days with this "
        "employer. Decide as you genuinely would.\n\n"
        'Respond with ONLY a JSON object: {"effort": <integer 0-10>}.'
    )


def parse_response(text: str) -> "dict | None":  # CHANGE HERE (3 of 4)
    """Extract the effort choice. Return None on failure — never guess."""
    match = re.search(r'\{\s*"effort"\s*:\s*(\d+)\s*\}', text)
    if match:
        effort = int(match.group(1))
        if 0 <= effort <= 10:
            return {"effort": effort}
    return None


def mock_response(prompt: str) -> str:           # CHANGE HERE (4 of 4)
    """DRY_RUN only: synthetic placeholder answers so the notebook executes
    without API calls. NOT model behavior — these numbers are made up and
    deliberately produce the double-match pattern (textbook piece-rate
    optimum + gift-exchange gradient) so the analysis tables are legible.
    Edit only if the answer format changes."""
    if "$2.50 for every unit" in prompt:
        center = 5.0                              # selfish optimum
    elif "fixed $25" in prompt:
        center = 6.5                              # gift-exchange response
    else:                                         # flat $10
        center = 2.0
    if "purely for the money" in prompt:          # money_focused
        center = 5.0 if "$2.50" in prompt else max(center - 1.5, 0.0)
    elif "doing right by people" in prompt:       # community_minded
        center = center + (1.0 if "fixed" in prompt else 0.0)
    effort = int(round(min(max(random.gauss(center, 1.0), 0), 10)))
    return json.dumps({"effort": effort})


## Run the experiment *(do not modify)*

Runs every persona × contract cell, logs parse failures instead of
dropping them, saves `lab4_results.csv` and `lab4_prompts_log.json`
(required in your write-up appendix). With `DRY_RUN = False` this makes
~270 API calls.


In [ ]:
def run_experiment() -> pd.DataFrame:
    keys = list(PERSONA_GRID)
    cells = [dict(zip(keys, combo)) for combo in itertools.product(*PERSONA_GRID.values())]
    print(f"{len(cells)} cells x {N_PER_CELL} agents = {len(cells) * N_PER_CELL} calls")

    rows = []
    for cell in cells:
        prompt = build_prompt(cell)
        for i in range(N_PER_CELL):
            try:
                response = client.messages.create(
                    model=MODEL,
                    max_tokens=100,
                    temperature=TEMPERATURE,
                    messages=[{"role": "user", "content": prompt}],
                )
                raw = response.content[0].text
                decision = parse_response(raw)
            except Exception as err:            # log failures; never drop silently
                raw, decision = f"ERROR: {err}", None
                time.sleep(5)
            rows.append({
                **cell, "rep": i,
                "effort": None if decision is None else decision["effort"],
                "raw": raw, "model": MODEL, "temperature": TEMPERATURE,
            })
        done = [r for r in rows if r["effort"] is not None]
        print(f"  cell {cell} done ({len(done)}/{len(rows)} parsed)")

    df = pd.DataFrame(rows)
    df.to_csv("lab4_results.csv", index=False)
    # Save exact prompts — required in the write-up appendix.
    with open("lab4_prompts_log.json", "w") as f:
        json.dump({str(c): build_prompt(c) for c in cells}, f, indent=2)
    return df


df = run_experiment()


## Analysis vs. the two benchmarks *(do not modify)*

Selfish benchmark: effort 0 under both flat wages, 5 under the piece rate
(the optimum of $2.5e - e^2/4$). Gift-exchange benchmark (FKR 1993):
effort **increasing** in flat-wage generosity. The gift-exchange gap
(flat-generous minus flat-low effort) is the headline statistic.


In [ ]:
SELFISH_OPTIMUM = {"flat_low": 0, "flat_generous": 0, "piece_rate": 5}

valid = df.dropna(subset=["effort"]).copy()
print(f"Parse-failure rate: {df['effort'].isna().mean():.3f} "
      "(report this in Section 4 of the write-up)")

print("\n=== Mean effort by persona x contract ===")
table = valid.pivot_table(index="persona", columns="contract",
                          values="effort", aggfunc="mean").round(2)
print(table)
print(f"\nSelfish benchmark by contract: {SELFISH_OPTIMUM}")

print("\n=== The gift-exchange gap (flat_generous - flat_low) ===")
flat = valid[valid["contract"].isin(["flat_low", "flat_generous"])]
gaps = flat.pivot_table(index="persona", columns="contract",
                        values="effort", aggfunc="mean")
gaps["gift_gap"] = (gaps["flat_generous"] - gaps["flat_low"]).round(2)
print(gaps.round(2))
print("Selfish theory: gap = 0. FKR humans: clearly positive.")

print("\n=== Piece-rate calibration ===")
pr = valid[valid["contract"] == "piece_rate"]["effort"]
print(f"Mean piece-rate effort: {pr.mean():.2f} (selfish optimum: 5). ")
print(f"Share choosing exactly 5: {(pr == 5).mean():.2f} — a very high "
      "share is the 'solving, not behaving' signature (PS4 Part C).")


## Robustness — required in every lab

Rerun at least the `baseline × flat_generous` and `baseline × piece_rate`
cells with:

1. **a paraphrased prompt** (rewrite `build_prompt`'s wording, same
   content);
2. **a second model** (change `MODEL` in the setup cell);
3. **the strip-the-framing run** — rewrite the scenario as a mundane
   shift-work vignette (no cost table typography, no experiment flavor,
   costs described in prose). If the gift-exchange gap vanishes, what was
   your baseline result made of?

**Limits of silicon subjects:** the model has read the textbook, FKR
(1993), and Gneezy & List (2006) — every pattern here has a recitation
explanation, and the double-match (textbook piece-rate optimum AND
gift-exchange gradient in one run) is grounds for suspicion, not double
validation. Silicon effort costs nothing: the cost table is words, not
disutility. Your divergence diagnosis (write-up Section 5) must take a
position on what determines which literature the agents reproduce —
and Section 6 must say what only a human study can establish.
